# Dis2Vec UPNA-DDAL Inference Notebook

This notebook demonstrates how to use the trained UPNA-DDAL model for disease-disease association predictions using Dis2Vec embeddings.

**Model**: `UPNA-UPNA-DDAL_epochs_2__embeddings_node2vec_MechDist_dimensions_128_WalkLength_10_NumWalks_100.h5`

**Embeddings**: 128-dimensional Node2Vec disease representations learned from the MechDist network

## ✅ Files Available

All required files are now in the `../data/` directory:

1. **✅ Model file** (33 MB):
   - `UPNA-DDAL_epochs_2__embeddings_node2vec_MechDist_dimensions_128_WalkLength_10_NumWalks_100.h5`

2. **✅ Embeddings file** (2.5 MB):
   - `embeddings_node2vec_MechDist_dimensions_128_WalkLength_10_NumWalks_100.pickle`
   - Contains 128-dimensional embeddings for 4,710 diseases

3. **✅ Disease names** (23 MB):
   - `disease_id_names.csv`
   - Maps disease IDs to human-readable names

## What This Notebook Does

- **Load** the trained UPNA-DDAL model and disease embeddings
- **Predict** association scores between any two diseases
- **Generate** new predictions not in the pre-computed file
- **Explore** top associations for specific diseases of interest

## Alternative: Query Pre-computed Predictions

If you prefer to query existing predictions instead of generating new ones:
- Use `query_dis2vec_predictions.ipynb` 
- Requires `dis2vec_all_preds.parquet.gzip` from [Zenodo](https://zenodo.org/records/21251215)
- Contains ~22 million pre-computed predictions

## Model Performance

- **AUROC**: 0.89 on IxIDN/ORDON benchmarks
- **Coverage**: 4,710 diseases (4,565 rare + 145 non-rare)
- **Task**: Predict therapeutic transferability between disease pairs

## 1. Setup and Imports

In [1]:
import numpy as np
import pandas as pd
import pickle
from keras.models import load_model
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print('Imports successful!')

Imports successful!


## 2. Configure Paths

Update these paths to match your local file locations.

In [2]:
# File paths - all files are in ../data/
MODEL_PATH = '../data/UPNA-DDAL_epochs_2__embeddings_node2vec_MechDist_dimensions_128_WalkLength_10_NumWalks_100.h5'
EMBEDDINGS_PATH = '../data/embeddings_node2vec_MechDist_dimensions_128_WalkLength_10_NumWalks_100.pickle'
DISEASE_NAMES_PATH = '../data/disease_id_names.csv'

print('File paths configured:')
print(f'  Model: {MODEL_PATH}')
print(f'  Embeddings: {EMBEDDINGS_PATH}')
print(f'  Disease names: {DISEASE_NAMES_PATH}')
print()
print('Note: Embeddings learned from MechDist (Mechanistic Distance) network')

File paths configured:
  Model: ../data/UPNA-DDAL_epochs_2__embeddings_node2vec_MechDist_dimensions_128_WalkLength_10_NumWalks_100.h5
  Embeddings: ../data/embeddings_node2vec_MechDist_dimensions_128_WalkLength_10_NumWalks_100.pickle
  Disease names: ../data/disease_id_names.csv

Note: Embeddings learned from MechDist (Mechanistic Distance) network


## 3. Load Model and Embeddings

In [3]:
# Load Dis2Vec embeddings
print('Loading embeddings...')
with open(EMBEDDINGS_PATH, 'rb') as handle:
    MechDist_embeddings = pickle.load(handle)

# Extract configuration
embedding_config = MechDist_embeddings['config']
node2vec_dimensions = embedding_config['dimensions']

print(f"✅ Embeddings loaded successfully!")
print(f"\nEmbedding configuration:")
print(f"  Dimensions: {embedding_config['dimensions']}")
print(f"  Walk length: {embedding_config['walk_length']}")
print(f"  Num walks: {embedding_config['num_walks']}")
print(f"  Window: {embedding_config['window']}")
print(f"\nTotal diseases with embeddings: {len(MechDist_embeddings) - 1}")  # -1 for 'config' key

Loading embeddings...
✅ Embeddings loaded successfully!

Embedding configuration:
  Dimensions: 128
  Walk length: 10
  Num walks: 100
  Window: 10

Total diseases with embeddings: 4710


In [4]:
# Load trained UPNA-DDAL model
print('Loading model...')
model = load_model(MODEL_PATH)
print('Model loaded successfully!')
print(f'\nModel architecture:')
model.summary()

Loading model...
Model loaded successfully!

Model architecture:


Model: "model_8"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_17            │ (None, 128)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_18            │ (None, 128)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_40 (Dense)    │ (None, 2048)      │    264,192 │ input_17[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_41 (Dense)    │ (None, 2048)      │    264,192 │ input_18[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_8       │ (None, 4096)      │          0 │ dense_40[0][0],   │
│ (Concatenate)       │                   │            │ dense_41[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_8 (Dropout) │ (None, 4096)      │          0 │ concatenate_8[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_42 (Dense)    │ (None, 512)       │  2,097,664 │ dropout_8[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_24       │ (None, 512)       │          0 │ dense_42[0][0]    │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_43 (Dense)    │ (None, 512)       │    262,656 │ activation_24[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_25       │ (None, 512)       │          0 │ dense_43[0][0]    │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_44 (Dense)    │ (None, 1)         │        513 │ activation_25[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_26       │ (None, 1)         │          0 │ dense_44[0][0]    │
│ (Activation)        │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,889,219 (11.02 MB)

 Trainable params: 2,889,217 (11.02 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [5]:
# Optional: Load disease names for interpretability
try:
    disease_id_names_df = pd.read_csv(DISEASE_NAMES_PATH)
    mapping_disease_id_name = disease_id_names_df.set_index('disease_id')['name'].to_dict()
    print(f'Loaded {len(mapping_disease_id_name)} disease names')
except FileNotFoundError:
    print('Disease names file not found. Predictions will show disease IDs only.')
    mapping_disease_id_name = {}

Loaded 35052 disease names


## 4. Prepare Node Mapping

The model operates on integer node IDs. We need to map between disease IDs (e.g., 'ORPHA_2134') and node IDs.

In [6]:
# Get all disease node IDs (excluding 'config' key)
diseases = [k for k in MechDist_embeddings.keys() if k != 'config']

print(f'Total diseases available for inference: {len(diseases)}')
print(f'\nExample disease IDs: {diseases[:5]}')

Total diseases available for inference: 4710

Example disease IDs: [0, 1, 2, 3, 4]


## 5. Inference Functions

In [7]:
def predict_disease_pair(disease_id_1, disease_id_2, model, embeddings, dim, return_stats=False):
    """
    Predict disease-disease association score for a single pair.
    Uses bidirectional prediction (both directions averaged) to match training.
    
    Args:
        disease_id_1: First disease node ID
        disease_id_2: Second disease node ID
        model: Trained Keras model
        embeddings: Dictionary of disease embeddings
        dim: Embedding dimension
        return_stats: If True, return (mean, std); if False, return mean only
    
    Returns:
        float or tuple: Mean score, or (mean, std) if return_stats=True
    """
    emb1 = embeddings[disease_id_1]
    emb2 = embeddings[disease_id_2]
    
    # Direction 1: (emb1, emb2)
    X1 = np.array([list(emb1) + list(emb2)])
    pred1 = model.predict([X1[:, :dim], X1[:, dim:]], verbose=0)[0][0]
    
    # Direction 2: (emb2, emb1) - swap arms
    X2 = np.array([list(emb2) + list(emb1)])
    pred2 = model.predict([X2[:, :dim], X2[:, dim:]], verbose=0)[0][0]
    
    # Compute mean and std
    pred_mean = (pred1 + pred2) / 2.0
    pred_std = np.std([pred1, pred2])
    
    if return_stats:
        return pred_mean, pred_std
    else:
        return pred_mean


def predict_all_pairs_for_disease(input_disease_id, model, embeddings, dim, disease_names=None):
    """
    Predict associations between input disease and all other diseases.
    Uses bidirectional prediction with mean and std.
    
    Args:
        input_disease_id: Target disease node ID
        model: Trained Keras model
        embeddings: Dictionary of disease embeddings
        dim: Embedding dimension
        disease_names: Optional dict mapping disease IDs to names
    
    Returns:
        DataFrame with columns: node1, node2, Dis2vec_Pred_mean, Dis2vec_Pred_std
    """
    all_diseases = [k for k in embeddings.keys() if k != 'config']
    
    results = []
    
    print(f'Predicting for {len(all_diseases)-1} disease pairs...')
    for dis in tqdm(all_diseases):
        if input_disease_id == dis:
            continue
        
        pred_mean, pred_std = predict_disease_pair(
            input_disease_id, dis, model, embeddings, dim, return_stats=True
        )
        
        result = {
            'node1': input_disease_id,
            'node2': dis,
            'Dis2vec_Pred_mean': pred_mean,
            'Dis2vec_Pred_std': pred_std
        }
        
        if disease_names:
            result['node1_name'] = disease_names.get(input_disease_id, 'N/A')
            result['node2_name'] = disease_names.get(dis, 'N/A')
        
        results.append(result)
    
    pred_df = pd.DataFrame(results).sort_values(by='Dis2vec_Pred_mean', ascending=False).reset_index(drop=True)
    
    return pred_df


print('Inference functions defined!')

Inference functions defined!


## 6. Example Predictions

### Example 1: Single Disease Pair Prediction

In [ ]:
# Example: Predict a few disease pairs
example_pairs = [
    (diseases[0], diseases[1]),
    (diseases[0], diseases[2]),
    (diseases[1], diseases[2]),
]

print("Disease pair predictions:\n")

for dis1, dis2 in example_pairs:
    score_mean, score_std = predict_disease_pair(
        dis1, dis2, model, MechDist_embeddings, node2vec_dimensions, return_stats=True
    )
    
    print(f"Pair: {dis1} ↔ {dis2}")
    if mapping_disease_id_name:
        print(f"  {mapping_disease_id_name.get(dis1, 'N/A')[:60]}")
        print(f"  {mapping_disease_id_name.get(dis2, 'N/A')[:60]}")
    print(f"  Mean: {score_mean:.6f}")
    print(f"  Std:  {score_std:.6f}")
    print()

Disease pair predictions:



### Example 2: Find Top Associations for a Disease

In [ ]:
# Find all associations for a target disease
target_disease = diseases[0]  # Replace with your disease of interest

print(f"Finding associations for: {target_disease}")
if mapping_disease_id_name:
    print(f"Disease name: {mapping_disease_id_name.get(target_disease, 'N/A')}")

results = predict_all_pairs_for_disease(
    target_disease,
    model,
    MechDist_embeddings,
    node2vec_dimensions,
    disease_names=mapping_disease_id_name if mapping_disease_id_name else None
)

print(f"\nTop 20 predicted associations:")
if 'node2_name' in results.columns:
    display(results[['node2', 'node2_name', 'Dis2vec_Pred_mean', 'Dis2vec_Pred_std']].head(20))
else:
    display(results[['node2', 'Dis2vec_Pred_mean', 'Dis2vec_Pred_std']].head(20))

## 7. Batch Prediction (Optional)

Generate predictions for specific disease pairs from a list.

In [ ]:
# Define disease pairs of interest
disease_pairs = [
    (diseases[0], diseases[1]),
    (diseases[2], diseases[3]),
    # Add more pairs as needed
]

batch_results = []

for dis1, dis2 in tqdm(disease_pairs, desc='Predicting pairs'):
    score_mean, score_std = predict_disease_pair(
        dis1, dis2, model, MechDist_embeddings, node2vec_dimensions, return_stats=True
    )
    
    result = {
        'disease_1': dis1,
        'disease_2': dis2,
        'score_mean': score_mean,
        'score_std': score_std
    }
    
    if mapping_disease_id_name:
        result['disease_1_name'] = mapping_disease_id_name.get(dis1, 'N/A')
        result['disease_2_name'] = mapping_disease_id_name.get(dis2, 'N/A')
    
    batch_results.append(result)

batch_df = pd.DataFrame(batch_results).sort_values('score_mean', ascending=False)
print("\nBatch prediction results:")
display(batch_df)

## 8. Export Results (Optional)

In [ ]:
# Export predictions to file
output_path = '../temp/my_disease_predictions.csv'

# Uncomment to save results
# results.to_csv(output_path, index=False)
# print(f'Results saved to: {output_path}')

## Notes

- **Model Architecture**: UPNA-DDAL is a two-shot learning model with a 3-layer MLP decoder
- **Input**: Concatenated 128-dimensional Dis2Vec embeddings for two diseases (256 total dimensions)
- **Output**: Association probability between 0 and 1
- **Interpretation**: 
  - High scores (>0.8): Strong predicted therapeutic transferability
  - Medium scores (0.5-0.8): Moderate association
  - Low scores (<0.5): Weak or no predicted association

- **Performance**: AUROC = 0.89 on IxIDN/ORDON benchmarks (cross-evidence generalization)

For more details, see the manuscript: "Representation Learning of Human Diseases for Indication Expansion and Investment Decisions"